In [1]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('intelligence', 'grade'),
    ('studyhours', 'grade'),
    ('difficulty', 'grade'),
    ('grade', 'pass')
])

cpd_i = TabularCPD(variable='intelligence', variable_card=2,
                   values=[[0.7], [0.3]])

cpd_s = TabularCPD(variable='studyhours', variable_card=2,
                   values=[[0.6], [0.4]])

cpd_d = TabularCPD(variable='difficulty', variable_card=2,
                   values=[[0.4], [0.6]])

cpd_g = TabularCPD(
    variable='grade',
    variable_card=3,
    values=[
        [0.60, 0.80, 0.30, 0.50, 0.20, 0.40, 0.05, 0.15],
        [0.30, 0.15, 0.40, 0.35, 0.40, 0.40, 0.25, 0.35],
        [0.10, 0.05, 0.30, 0.15, 0.40, 0.20, 0.70, 0.50]
    ],
    evidence=['intelligence', 'studyhours', 'difficulty'],
    evidence_card=[2, 2, 2]
)

cpd_p = TabularCPD(
    variable='pass',
    variable_card=2,
    values=[
        [0.95, 0.80, 0.50],
        [0.05, 0.20, 0.50]
    ],
    evidence=['grade'],
    evidence_card=[3]
)

model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
assert model.check_model()

infer = VariableElimination(model)

r1 = infer.query(variables=['pass'], evidence={'studyhours': 0, 'difficulty': 0})
print("p(pass | studyhours=sufficient, difficulty=hard):")
print(r1)

r2 = infer.query(variables=['intelligence'], evidence={'pass': 0})
print("p(intelligence | pass=yes):")
print(r2)

c:\Users\Mohammad Burair\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


p(pass | studyhours=sufficient, difficulty=hard):
+---------+-------------+
| pass    |   phi(pass) |
+=========+=============+
| pass(0) |      0.8150 |
+---------+-------------+
| pass(1) |      0.1850 |
+---------+-------------+
p(intelligence | pass=yes):
+-----------------+---------------------+
| intelligence    |   phi(intelligence) |
+=================+=====================+
| intelligence(0) |              0.7354 |
+-----------------+---------------------+
| intelligence(1) |              0.2646 |
+-----------------+---------------------+


In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('disease', 'fever'),
    ('disease', 'cough'),
    ('disease', 'fatigue'),
    ('disease', 'chills')
])

cpd_dis = TabularCPD(variable='disease', variable_card=2,
                     values=[[0.3], [0.7]])

cpd_fev = TabularCPD(
    variable='fever',
    variable_card=2,
    values=[
        [0.9, 0.5],
        [0.1, 0.5]
    ],
    evidence=['disease'],
    evidence_card=[2]
)

cpd_cou = TabularCPD(
    variable='cough',
    variable_card=2,
    values=[
        [0.8, 0.6],
        [0.2, 0.4]
    ],
    evidence=['disease'],
    evidence_card=[2]
)

cpd_fat = TabularCPD(
    variable='fatigue',
    variable_card=2,
    values=[
        [0.7, 0.3],
        [0.3, 0.7]
    ],
    evidence=['disease'],
    evidence_card=[2]
)

cpd_chi = TabularCPD(
    variable='chills',
    variable_card=2,
    values=[
        [0.6, 0.4],
        [0.4, 0.6]
    ],
    evidence=['disease'],
    evidence_card=[2]
)

model.add_cpds(cpd_dis, cpd_fev, cpd_cou, cpd_fat, cpd_chi)
assert model.check_model()

infer = VariableElimination(model)

r1 = infer.query(variables=['disease'], evidence={'fever': 0, 'cough': 0})
print("p(disease | fever=yes, cough=yes):")
print(r1)

r2 = infer.query(variables=['disease'], evidence={'fever': 0, 'cough': 0, 'chills': 0})
print("p(disease | fever=yes, cough=yes, chills=yes):")
print(r2)

r3 = infer.query(variables=['fatigue'], evidence={'disease': 0})
print("p(fatigue=yes | disease=flu):")
print(r3)

p(disease | fever=yes, cough=yes):
+------------+----------------+
| disease    |   phi(disease) |
+============+================+
| disease(0) |         0.5070 |
+------------+----------------+
| disease(1) |         0.4930 |
+------------+----------------+
p(disease | fever=yes, cough=yes, chills=yes):
+------------+----------------+
| disease    |   phi(disease) |
+============+================+
| disease(0) |         0.6067 |
+------------+----------------+
| disease(1) |         0.3933 |
+------------+----------------+
p(fatigue=yes | disease=flu):
+------------+----------------+
| fatigue    |   phi(fatigue) |
+============+================+
| fatigue(0) |         0.7000 |
+------------+----------------+
| fatigue(1) |         0.3000 |
+------------+----------------+


In [1]:
import numpy as np

states = ["sunny", "cloudy", "rainy"]

tm = np.array([
    [0.6, 0.3, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

def simulate(init, steps):
    curr = init
    seq = [curr]
    for _ in range(steps):
        idx = states.index(curr)
        nxt = np.random.choice(states, p=tm[idx])
        seq.append(nxt)
        curr = nxt
    return seq

def rainy_prob(init, steps, min_rainy, simulations=10000):
    count = 0
    for _ in range(simulations):
        seq = simulate(init, steps)
        if seq.count("rainy") >= min_rainy:
            count += 1
    return count / simulations

seq = simulate("sunny", 10)
print("weather sequence for 10 days:")
print(" -> ".join(seq))

prob = rainy_prob("sunny", 10, 3)
print(f"p(at least 3 rainy days in 10 days): {prob:.4f}")

weather sequence for 10 days:
sunny -> sunny -> cloudy -> rainy -> rainy -> rainy -> rainy -> cloudy -> sunny -> sunny -> cloudy
p(at least 3 rainy days in 10 days): 0.4518
